In [1]:
import pandas as pd

url = "https://www.nikkansports.com/baseball/professional/score/2026/pl2026051004.html"

tables = pd.read_html(url)

print("表の数:", len(tables))

for i, table in enumerate(tables):
    print("=" * 50)
    print("table番号:", i)
    print(table.head())
    print(table.shape)

表の数: 4
table番号: 0
    守備 選手名 Unnamed: 2 打数 得点 安打 打点    通算率 ＨＲ   １   ２   ３   ４   ５   ６   ７   ８  \
0  (右)  佐藤        （右）  ４  ０  ０  ０  0.313  ２  遊ゴ  ……  遊ゴ  ……  三直  ……  ……  二飛   
1  (中)  辰己        （左）  ４  ０  １  ０  0.298  ３  一ゴ  ……  二ゴ  ……  ……  三振  ……  左安   
2  (遊)  村林        （右）  ４  ０  ２  ０  0.258  ３  中安  ……  ……  二ゴ  ……  右安  ……  三ゴ   
3  (指)  浅村        （右）  ３  ０  ０  ０  0.265  ４  四球  ……  ……  三振  ……  三振  ……  ……   
4  (一)  黒川        （左）  ３  ０  ０  ０  0.244  １  二ゴ  ……  ……  三直  ……  遊飛  ……  ……   

     ９  
0  NaN  
1  NaN  
2  NaN  
3   三ゴ  
4  NaN  
(12, 18)
table番号: 1
   チーム  １  ２  ３  ４  ５  ６  ７  ８  ９  計
0  楽 天  0  0  0  0  1  0  0  0  0  １
1  西 武  1  1  1  2  0  1  0  0  X  ６
(2, 11)
table番号: 2
    守備  選手名 Unnamed: 2 打数 得点 安打 打点    通算率 ＨＲ   １   ２   ３   ４   ５   ６   ７   ８  \
0  (右)  長谷川        （右）  ４  １  １  ０  0.259  ４  死球  左安  ……  三ゴ  ……  捕邪  ……  三ゴ   
1  (中)   西川        （左）  ３  １  １  ３  0.218  １  中飛  中犠  ……  右本  ……  三邪  ……  ……   
2  (指)   渡部        （右）  ４  ０  １  １  0.278  ４  左飛  右飛  ……  右飛  …

In [2]:
import pandas as pd
import re

url = "https://www.nikkansports.com/baseball/professional/score/2026/pl2026051004.html"

tables = pd.read_html(url)

rakuten_table = tables[0]
seibu_table = tables[2]

game_id = "2026051004"
date = "2026-05-10"

def batting_table_to_long(df, team, top_bottom):
    rows = []

    inning_cols = ["１", "２", "３", "４", "５", "６", "７", "８", "９"]

    for idx, row in df.iterrows():
        batter_order = idx + 1
        position = row["守備"]
        batter = row["選手名"]

        for inn_col in inning_cols:
            if inn_col not in df.columns:
                continue

            result = row[inn_col]

            # 空欄・……は打席なしなので除外
            if pd.isna(result):
                continue

            result = str(result).strip()

            if result == "……":
                continue

            # 全角数字を半角に変換
            inning = int(str(inn_col).translate(str.maketrans("１２３４５６７８９", "123456789")))

            rows.append({
                "game_id": game_id,
                "date": date,
                "team": team,
                "top_bottom": top_bottom,
                "batter_order": batter_order,
                "position": position,
                "batter": batter,
                "inning": inning,
                "play_result": result
            })

    return pd.DataFrame(rows)

rakuten_long = batting_table_to_long(rakuten_table, "楽天", "表")
seibu_long = batting_table_to_long(seibu_table, "西武", "裏")

batting_long = pd.concat([rakuten_long, seibu_long], ignore_index=True)

# 回・表裏・打順で並べ替え
top_bottom_order = {"表": 0, "裏": 1}
batting_long["top_bottom_order"] = batting_long["top_bottom"].map(top_bottom_order)

batting_long = batting_long.sort_values(
    ["inning", "top_bottom_order", "batter_order"]
).drop(columns=["top_bottom_order"])

batting_long.to_csv("batting_results_long.csv", index=False, encoding="utf-8-sig")

print(batting_long.head(30))
print("保存しました: batting_results_long.csv")

       game_id        date team top_bottom  batter_order position batter  \
0   2026051004  2026-05-10   楽天          表             1      (右)     佐藤   
4   2026051004  2026-05-10   楽天          表             2      (中)     辰己   
8   2026051004  2026-05-10   楽天          表             3      (遊)     村林   
12  2026051004  2026-05-10   楽天          表             4      (指)     浅村   
16  2026051004  2026-05-10   楽天          表             5      (一)     黒川   
34  2026051004  2026-05-10   楽天          表            12        計      計   
43  2026051004  2026-05-10   西武          裏             1      (右)    長谷川   
48  2026051004  2026-05-10   西武          裏             2      (中)     西川   
52  2026051004  2026-05-10   西武          裏             3      (指)     渡部   
56  2026051004  2026-05-10   西武          裏             4      (一)    ネビン   
60  2026051004  2026-05-10   西武          裏             5      (左)    林安可   
64  2026051004  2026-05-10   西武          裏             7      (三)     平沢   
80  20260510